In [2]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Makes sure we get same data every time
np.random.seed(42)
random.seed(42)

NUM_ROWS = 20000

# Things to pick from randomly — like real GL data
gl_accounts    = ['1001-Cash','1200-AR','2001-AP','3001-Equity',
                  '4001-Revenue','5001-COGS','6001-OpEx','7001-Tax']
descriptions   = ['Vendor Payment','Customer Receipt','Payroll','Accrual',
                  'Intercompany Transfer','Adjustment','Reversal','Depreciation']
users          = ['u.sharma','a.mehta','r.das','p.iyer','admin','s.roy','m.khan']
business_units = ['Finance','Operations','HR','IT','Sales']
currencies     = ['INR','USD','CAD']
entry_types    = ['Auto','Manual']

start_date = datetime(2023, 1, 1)

# ── BUILD BASE DATA ──────────────────────────────────────────
data = {
    'JournalID'    : [f'JE{str(i).zfill(6)}' for i in range(1, NUM_ROWS+1)],
    'EntryDate'    : [start_date + timedelta(days=random.randint(0,364),
                      hours=random.randint(0,23), minutes=random.randint(0,59))
                      for _ in range(NUM_ROWS)],
    'GLAccount'    : [random.choice(gl_accounts)    for _ in range(NUM_ROWS)],
    'Description'  : [random.choice(descriptions)   for _ in range(NUM_ROWS)],
    'Amount'       : [round(random.uniform(100,100000),2) for _ in range(NUM_ROWS)],
    'Currency'     : [random.choice(currencies)     for _ in range(NUM_ROWS)],
    'UserID'       : [random.choice(users)          for _ in range(NUM_ROWS)],
    'EntryType'    : [random.choice(entry_types)    for _ in range(NUM_ROWS)],
    'BusinessUnit' : [random.choice(business_units) for _ in range(NUM_ROWS)],
}

df = pd.DataFrame(data)
df['EffectiveDate'] = df['EntryDate']

print(f"Base data created: {len(df)} rows")

Base data created: 20000 rows


In [4]:
# ── INJECT ANOMALIES ─────────────────────────────────────────
# These are the "bad" entries our Alteryx flags will catch later

# 1. ROUND NUMBERS (suspicious amounts like 50,000 exactly)
round_idx = random.sample(range(NUM_ROWS), 300)
for i in round_idx:
    df.loc[i,'Amount'] = random.choice([5000,10000,25000,50000,100000])

# 2. WEEKEND ENTRIES (posted on Saturday or Sunday)
used = set(round_idx)
weekend_idx = random.sample(list(set(range(NUM_ROWS))-used), 200)
for i in weekend_idx:
    base = start_date + timedelta(days=random.randint(0,364))
    days_ahead = (5 - base.weekday()) % 7
    sat = base + timedelta(days=days_ahead)
    df.loc[i,'EntryDate'] = sat.replace(hour=random.randint(9,17),
                                         minute=random.randint(0,59))

# 3. AFTER-HOURS ENTRIES (posted between 8PM and 6AM)
used |= set(weekend_idx)
ah_idx = random.sample(list(set(range(NUM_ROWS))-used), 250)
for i in ah_idx:
    base = start_date + timedelta(days=random.randint(0,364))
    hour = random.choice(list(range(20,24))+list(range(0,6)))
    df.loc[i,'EntryDate'] = base.replace(hour=hour,
                                          minute=random.randint(0,59))

# 4. DUPLICATE JOURNAL IDs (same ID used twice — a red flag)
used |= set(ah_idx)
dup_idx = random.sample(list(set(range(NUM_ROWS))-used), 100)
for i in dup_idx:
    df.loc[i,'JournalID'] = df.loc[random.choice(
        list(set(range(NUM_ROWS))-{i})),'JournalID']

print(" Anomalies injected:")
print(f"   Round numbers   → {len(round_idx)} rows")
print(f"   Weekend entries → {len(weekend_idx)} rows")
print(f"   After-hours     → {len(ah_idx)} rows")
print(f"   Duplicate IDs   → {len(dup_idx)} rows")

 Anomalies injected:
   Round numbers   → 300 rows
   Weekend entries → 200 rows
   After-hours     → 250 rows
   Duplicate IDs   → 100 rows


In [6]:
save_path = r'C:\Users\Rim\Downloads\AuditLens\Data\gl_data.csv'

df.to_csv(save_path, index=False)
print(f" File saved successfully!")
print(f"   Location : {save_path}")
print(f"   Rows     : {len(df)}")
print(f"   Columns  : {list(df.columns)}")

 File saved successfully!
   Location : C:\Users\Rim\Downloads\AuditLens\Data\gl_data.csv
   Rows     : 20000
   Columns  : ['JournalID', 'EntryDate', 'GLAccount', 'Description', 'Amount', 'Currency', 'UserID', 'EntryType', 'BusinessUnit', 'EffectiveDate']


In [7]:
# ── QUICK VERIFICATION ───────────────────────────────────────
print("=== SHAPE (should be 20000 rows, 10 columns) ===")
print(df.shape)

print("\n=== COLUMNS ===")
print(df.columns.tolist())

print("\n=== FIRST 3 ROWS ===")
print(df.head(3).to_string())

print("\n=== ANOMALY SUMMARY ===")
print(f"Round amounts (÷1000):  {(df['Amount'] % 1000 == 0).sum()} rows")
print(f"Manual entries:         {(df['EntryType']=='Manual').sum()} rows")
print(f"Duplicate JournalIDs:   {df['JournalID'].duplicated().sum()} rows")

=== SHAPE (should be 20000 rows, 10 columns) ===
(20000, 10)

=== COLUMNS ===
['JournalID', 'EntryDate', 'GLAccount', 'Description', 'Amount', 'Currency', 'UserID', 'EntryType', 'BusinessUnit', 'EffectiveDate']

=== FIRST 3 ROWS ===
  JournalID           EntryDate  GLAccount            Description    Amount Currency UserID EntryType BusinessUnit       EffectiveDate
0  JE000001 2023-11-24 03:01:00    2001-AP  Intercompany Transfer  96561.65      CAD  admin      Auto   Operations 2023-11-24 03:01:00
1  JE000002 2023-05-21 07:14:00  5001-COGS                Accrual  46491.08      USD  admin      Auto           IT 2023-05-21 07:14:00
2  JE000003 2023-03-13 23:06:00  1001-Cash             Adjustment  65096.71      USD  admin      Auto        Sales 2023-03-13 23:06:00

=== ANOMALY SUMMARY ===
Round amounts (÷1000):  595 rows
Manual entries:         9971 rows
Duplicate JournalIDs:   198 rows


In [8]:
print("=== SHAPE (should be 20000 rows, 10 columns) ===")
print(df.shape)

=== SHAPE (should be 20000 rows, 10 columns) ===
(20000, 10)
